In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats

# 1. CARGA Y PROCESAMIENTO DE DATOS (El Source)
df = pd.read_csv('datos_loja2.csv')

# Calcular la variable derivada: Porcentaje de viviendas sin alcantarillado
df['Porcentaje_Sin_Alcantarillado'] = (df['Sin_Alcantarillado'] / df['Viviendas']) * 100

print("--- DATASET PROCESADO ---")
print(df[['Canton', 'Poblacion', 'Porcentaje_Sin_Alcantarillado']].head())


# =====================================================================
# PUNTO 1: PRUEBA DE HIPÓTESIS UNIMUESTRAL (T-STUDENT)
# =====================================================================
# Parámetro de comparación (Mu_0 = 32%)
mu_0 = 32.0

# Ejecutar el test T para una muestra (Alternative 'greater' para una cola)
t_stat, p_val_uni = stats.ttest_1samp(df['Porcentaje_Sin_Alcantarillado'], popmean=mu_0, alternative='greater')

print("\n--- PUNTO 1: TEST UNIMUESTRAL ---")
print(f"Estadístico T: {t_stat:.4f}")
print(f"Valor p: {p_val_uni:.4f}")


# =====================================================================
# PUNTO 2: COMPARACIÓN DE GRUPOS (A/B TESTING)
# =====================================================================
# Dividir el dataset en dos grupos según la mediana de población (~15,000 hab)
grupo_A = df[df['Poblacion'] >= 15000]['Porcentaje_Sin_Alcantarillado']
grupo_B = df[df['Poblacion'] < 15000]['Porcentaje_Sin_Alcantarillado']

# Ejecutar test T para muestras independientes (Asumiendo varianzas iguales o Welch)
t_stat_ind, p_val_ind = stats.ttest_ind(grupo_A, grupo_B, equal_var=False)

print("\n--- PUNTO 2: COMPARACIÓN DE GRUPOS (A/B Testing) ---")
print(f"Media Grupo A (>=15k hab): {grupo_A.mean():.2f}%")
print(f"Media Grupo B (<15k hab): {grupo_B.mean():.2f}%")
print(f"Estadístico T (independiente): {t_stat_ind:.4f}")
print(f"Valor p (bidireccional): {p_val_ind:.4f}")

### 📝 Formalismo Matemático: Prueba de Hipótesis Unimuestral

Definimos formalmente nuestras hipótesis con respecto a la media poblacional $\mu$ del porcentaje de viviendas sin alcantarillado sanitario en la provincia de Loja:

$$H_0: \mu \le 32.0\%$$
$$H_1: \mu > 32.0\%$$

Dado que trabajamos con una muestra pequeña ($n = 16$), empleamos el estadístico **T de Student** bajo la siguiente estructura matemática:

$$t = \frac{\bar{x} - \mu_0}{\frac{s}{\sqrt{n}}}$$

Donde:
* $\bar{x}$: Media muestral calculada de los cantones.
* $\mu_0$: Parámetro crítico de comparación ($32.0\%$).
* $s$: Desviación estándar muestral.
* $n$: Tamaño de la muestra ($16$ cantones).

In [7]:
import pandas as pd
import scipy.stats as stats

# Carga de datos
df = pd.read_csv('datos_loja2.csv')
df['Porcentaje_Sin_Alcantarillado'] = (df['Sin_Alcantarillado'] / df['Viviendas']) * 100

# Punto 1: Test Unimuestral (Media vs 32%)
mu_0 = 32.0
t_stat, p_val_uni = stats.ttest_1samp(df['Porcentaje_Sin_Alcantarillado'], popmean=mu_0, alternative='greater')

print(f"Media Muestral Calculada: {df['Porcentaje_Sin_Alcantarillado'].mean():.2f}%")
print(f"Estadístico T: {t_stat:.4f}")
print(f"Valor p: {p_val_uni:.4f}")

Media Muestral Calculada: 34.65%
Estadístico T: 1.5897
Valor p: 0.0664


### 📝 Formalismo Matemático: Muestras Independientes (A/B Testing)

Evaluamos si existe una diferencia estadísticamente significativa entre las medias de los dos grupos definidos por el volumen poblacional de los cantones ($\mu_A$ para cantones grandes/medianos y $\mu_B$ para cantones pequeños):

$$H_0: \mu_A = \mu_B$$
$$H_1: \mu_A \neq \mu_B$$

Utilizamos la prueba **T de Welch** para dos muestras independientes, la cual es robusta frente a tamaños de muestra desiguales y no asume homogeneidad de varianzas:

$$t = \frac{\bar{x}_A - \bar{x}_B}{\sqrt{\frac{s_A^2}{n_A} + \frac{s_B^2}{n_B}}}$$

In [9]:
# Punto 2: Comparación de Grupos (Grandes vs Pequeños)
grupo_A = df[df['Poblacion'] >= 15000]['Porcentaje_Sin_Alcantarillado']
grupo_B = df[df['Poblacion'] < 15000]['Porcentaje_Sin_Alcantarillado']

t_stat_ind, p_val_ind = stats.ttest_ind(grupo_A, grupo_B, equal_var=False)

print(f"Media Cantones Grandes (>=15k hab): {grupo_A.mean():.2f}%")
print(f"Media Cantones Pequeños (<15k hab): {grupo_B.mean():.2f}%")
print(f"Estadístico T de Welch: {t_stat_ind:.4f}")
print(f"Valor p (Bidireccional): {p_val_ind:.4f}")

Media Cantones Grandes (>=15k hab): 32.97%
Media Cantones Pequeños (<15k hab): 35.95%
Estadístico T de Welch: -0.7780
Valor p (Bidireccional): 0.4646
